# AlphaFold2 Ablation Study — Local Plotting (v2)

Enhanced copy of `plot_all_windows.ipynb`. Same `combine_plots` pipeline, but with
extra presentation controls: **legend layout**, **custom tick grid**, **axis titles**,
**per-element font sizes**, and **custom colours**. See the *Parameter reference* cell below.

No `uv` or shell commands needed — everything runs as pure Python.

In [1]:
import os, sys

# Make sure we're in the project root (not the notebooks folder)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Add the scripts folder to the Python path so we can import from it
scripts_dir = os.path.join(os.getcwd(), 'scripts')
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

print('Working directory:', os.getcwd())
print('Scripts dir on path:', scripts_dir)

Working directory: c:\Users\franc\OneDrive - TUM\2 - Protein Pred\Code\alphafold2-ablation-study
Scripts dir on path: c:\Users\franc\OneDrive - TUM\2 - Protein Pred\Code\alphafold2-ablation-study\scripts


In [2]:
# --- CONFIGURE THESE PATHS ---

# Where your experiments are
result_path = r"C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output"
# Where your repository code and 'data' folder is
base_repo_path = r"C:\Users\franc\OneDrive - TUM\2 - Protein Pred\Code\alphafold2-ablation-study"

print('Input folder:', base_repo_path)
print('Output folder:', result_path)
print('Input exists:', os.path.isdir(base_repo_path))
print('Output exists:', os.path.isdir(result_path))

Input folder: C:\Users\franc\OneDrive - TUM\2 - Protein Pred\Code\alphafold2-ablation-study
Output folder: C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output
Input exists: True
Output exists: True


## Parameter reference for `combine_plots`

`combine_plots(data_files, ...)` merges several per-experiment CSVs into one scatter and calls
`plot_tm_score`. **Every styling parameter below also exists on `plot_tm_score`** (use that
directly if you only plot a single CSV). Parameters marked **(NEW)** were added in v2.

### Core
- **`data_files`** (list[str]) — the CSVs to merge into one plot.
- **`save_file_name`** (str) — output PNG file name.
- **`output_dir`** (str) — folder to write the PNG (and the merged CSV).
- **`protein`** (str, default `None`) — needed only if a CSV holds more than one protein.
- **`title`** (str) — overrides the auto title `"{protein} | Depth: {depths}"`.
- **`experiment_name`** (str) — prepended to the auto title.

### Data → visual encoding
- **`color_on`** (str, default `"nseq"`) — column mapped to point **colour**:
  - `"nseq"` (MSA depth) → depth **colour bar** (default);
  - a **text** column (e.g. `"experiment"`) → discrete **colour legend**;
  - a **numeric** non-`nseq` column (e.g. `"seed"`) → numeric colour bar.
- **`shape_on`** (str) — column mapped to marker **shape** (discrete shape legend). Can be combined with `color_on`.
- **`opacity`** (float, 0–1) — point transparency.

### Colours **(NEW)**
- **`colors`** (list) — custom point colours for a categorical colour legend (`color_on` = a text column).
  Accepts any Matplotlib colour (hex, name, RGB tuple). **Aligned to the sorted unique category order**
  (same ordering rule as `legend_labels`). Cycles if you pass fewer colours than categories.
  Defaults to the Okabe–Ito colour-blind-safe palette.
- Helper: `from plot_tmscore import get_okabe_ito_colors` returns the palette as hex strings, so you can
  **mix palette + custom**, e.g. `colors=[pal[0], "#000000"]`.

### Legend
- **`legend_title`** (str) — legend heading text.
- **`legend_labels`** (list) — text for each legend entry, **aligned to sorted category order**.
- **`legend_layout`** (str, **NEW**, default `"auto"`) — `"auto"`/`"row"` = one horizontal row; `"column"` = stacked vertically.
- **`legend_title_fontsize`** (float, **NEW**) — absolute pt for the legend **title**.
- **`legend_text_fontsize`** (float, **NEW**) — absolute pt for the legend **entries**.

### Axes
- **`limit_axis`** (bool) — apply axis limits.
- **`axis_bounds`** (`[x_lo, x_hi, y_lo, y_hi]`) — explicit limits (requires `limit_axis=True`).
- **`x_axis_title`** / **`y_axis_title`** (str, **NEW**) — replace the auto `"Similarity to … (TM-score)"` labels.
- **`axis_title_fontsize`** (float, **NEW**) — absolute pt for **both** axis titles.
- **`tick_anchor`** (float, **NEW**) — a tick value the grid is aligned to (e.g. `0.95`).
- **`tick_size`** (float, **NEW**) — spacing between ticks (e.g. `0.05`). With `tick_anchor`, ticks are placed at
  `anchor + k·size` for **every** integer `k` that lands inside the axis range — **both axes, identical settings**.
  Both must be set to take effect; otherwise the default (~6 auto ticks) is used.
- **`plot_guidelines`** (bool) — dashed IF/OF (or active/inactive) reference lines.

### Fonts
- **`font_size`** (int, default 6) — global base size. The title, tick labels, and any **unset** size below are
  derived as `font_size + N`.
- The three **NEW** `*_fontsize` args are **absolute point sizes** and override that derivation for their element;
  leave them unset to keep the `font_size`-relative defaults.

> **Alignment tip — read this.** `legend_labels` and `colors` are assigned to the point groups in the
> **sorted order of the `experiment` column values** (the full paths baked into each CSV, *not* the folder
> names). That is **not** necessarily the order of your `experiments` list. Always confirm the order before
> trusting the labels:
>
> ```python
> import pandas as pd, numpy as np
> combined = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
> for i, c in enumerate(np.sort(combined["experiment"].unique())):
>     print(i, c)
> ```
>
> Then order `legend_labels` / `colors` so index *i* matches category *i*. (For this dataset the sorted order
> is `depth_5120`, `query_mask_15`, `query_mask_15_Seed1`, `query_mask_15_Seed2` — note `query_mask_15` and
> `query_mask_15_Seed7` are the **same** base experiment, seed 7 just left unspecified.)

In [3]:
# import os
# from plot_tmscore import combine_plots, get_okabe_ito_colors

# # --- experiments (folder names under plots/TM_Score) ---
# experiments = [
#     "depth_5120",
#     "query_mask_15_Seed1",
#     "query_mask_15_Seed2",
#     "query_mask_15_Seed7",
# ]

# protein_name = "MCT1"

# csv_files = [
#     os.path.join(result_path, "plots", "TM_Score", exp, f"{protein_name}.csv")
#     for exp in experiments
# ]

# # Okabe-Ito palette (colour-blind safe). Mix palette entries with your own hex colours.
# pal = get_okabe_ito_colors()

# combine_plots(
#     data_files=csv_files,
#     save_file_name=f"combined_{protein_name}_v2.png",
#     limit_axis=True,
#     axis_bounds=[0.7, 1.0, 0.7, 1.0],

#     # --- colour points by experiment (discrete colour legend) ---
#     color_on="experiment",
#     colors=[pal[1], pal[2], pal[3], "#000000"],       # SORTED-order aligned; mix palette + custom hex

#     # --- legend ---
#     legend_title="QM Seeds",
#     legend_labels=["No QM", "QM Seed 0", "QM Seed 1", "QM Seed 3"],  # sorted-order aligned
#     legend_layout="column",                           # "auto" | "row" | "column"
#     legend_title_fontsize=11,
#     legend_text_fontsize=9,

#     # --- custom tick grid (x & y): ticks at 0.95 +/- k*0.05 within the axis range ---
#     tick_anchor=0.95,
#     tick_size=0.05,

#     # --- axis titles ---
#     x_axis_title="TM-score to inward-facing state",
#     y_axis_title="TM-score to outward-facing state",
#     axis_title_fontsize=10,

#     output_dir=os.path.join(result_path, "plots", "TM_Score"),
# )

In [ ]:
import os
from plot_tmscore import combine_plots

experiments = [
    "depth_5120",
    "query_mask_15_Seed1",
    "query_mask_15_Seed2",
    "query_mask_15_Seed7",
]
protein_name = "MCT1"
csv_files = [
    os.path.join(result_path, "plots", "TM_Score", exp, f"{protein_name}.csv")
    for exp in experiments
]

combine_plots(
    data_files=csv_files,
    save_file_name=f"combined_{protein_name}_v2.png",
    limit_axis=True,
    axis_bounds=[0.7, 1.0, 0.7, 1.0],

    color_on="experiment",
    legend_title="QM Seeds",
    legend_labels=["No QM", "QM Seed 0", "QM Seed 1", "QM Seed 3"],

    # --- custom colours = the current standard Okabe–Ito assignment (by SORTED category) ---
    colors=[
        "#56b4e9",  # cat 0: depth_5120            -> "No QM"     | Okabe–Ito[0] = black
        "#e69f00",  # cat 1: query_mask_15 (Seed7) -> "QM Seed 0" | Okabe–Ito[1] = orange
        "#000000",  # cat 2: query_mask_15_Seed1   -> "QM Seed 1" | Okabe–Ito[2] = sky blue
        "#009e73",  # cat 3: query_mask_15_Seed2   -> "QM Seed 3" | Okabe–Ito[3] = bluish green
    ],

    # --- axis titles ---
    x_axis_title="TM-Score: Pred vs IF Conf.",
    y_axis_title="TM-Score: Pred vs OF Conf.",

    # --- ticks: step 0.05, aligned so the top tick is 1.0 (grid steps down 1.0, 0.95, ...) ---
    tick_anchor=1.0,
    tick_size=0.05,

    # --- every configurable font size, set to the current defaults (font_size=6 base) ---
    font_size=6,                 # base size (title, tick labels) — current default
    axis_title_fontsize=7,       # default = font_size + 1
    legend_text_fontsize=8,      # default = font_size + 2
    legend_title_fontsize=9,     # default = font_size + 3

    output_dir=os.path.join(result_path, "plots", "TM_Score"),
)


Saved COMBINED file to C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\TM_Score\depth_5120\MCT1_COMBINED_07_21_00_11.csv


### Minimal / backward-compatible example

Every new parameter is optional. Omitting them reproduces the original behaviour
(depth colour bar or plain colour legend, auto axis labels, ~6 auto ticks, `font_size`-relative sizes).
The call below is the pre-v2 style — uncomment to run.

In [5]:
# combine_plots(
#     data_files=csv_files,
#     save_file_name=f"combined_{protein_name}.png",
#     limit_axis=True,
#     axis_bounds=[0.7, 1.0, 0.7, 1.0],
#     color_on="experiment",
#     legend_title="QM Seeds",
#     legend_labels=["No QM", "QM Seed 0", "QM Seed 1", "QM Seed 3"],
#     output_dir=os.path.join(result_path, "plots", "TM_Score"),
# )